[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/quadric_collision/quadric_collision.ipynb)

# Collision of Two Ellipsoids

This example decides whether two ellipsoids are apart, touching or overlapping, with one number. Between the two shapes runs a family of blends, a little of one and the rest of the other, from the first to the second. If the ellipsoids are apart, some blend in between is a hyperboloid whose two branches part them, and a plane between them separates the pair; if they overlap, every blend in between stays an ellipsoid bridging the two. One number along the blends, peaking somewhere between, tells which: above zero apart, zero touching, below zero overlapping. At the touching pose that blend gives the contact itself: the shared tangent plane and the point where both ellipsoids meet it.

Points are antivectors and planes vectors, and an ellipsoid is held by its tangent planes, a dual quadric that maps each plane to a point, `Point <- Plane`: zero on the planes that touch it, `plane & dual(plane) == 0`, and sending each tangent to its point of contact. A motor carries the whole map, `motor >> dual(motor << Plane)`. Blends of two such maps keep their common tangents, and the number is the blend's determinant, a cubic in how far the blend has gone: four samples fix it, and only finding its peak leaves the algebra. At contact the blend at the peak is singular, and its null plane is the shared tangent.

In matrix notation the dual ellipsoids read as symmetric $3\times3$ matrices $C_1^*$ and $C_2^*$, the blends as the pencil $(1-\lambda)C_1^* + \lambda C_2^*$, and the test as the sign of the maximum of $\det$ over the pencil.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from numga import NumpyContext
from numga.algebras import PGA2D as ga

from examples.util.animation import show_animation
from examples.quadrics.quadric_collision import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Plane = ga.gatype.vector()                          # Vector+
Point = ga.gatype.antivector()                      # Antivector-
Motor = ga.gatype.rotor()                           # Rotor+
# The dual quadric maps a tangent plane to its contact point; the primal one maps a point to
# its polar plane.
Quadric = ga.gatype((Point, Plane))                 # Point- <- Plane+
Polarity = ga.gatype((Plane, Point))                # Plane+ <- Point-
# The plane at infinity, the origin, and the directions along x and y.
infinity = mv.w                                     # [] Plane+
origin = mv.point([0.0, 0.0])                       # [] Point-
along_x, along_y = mv.direction([1.0, 0.0]), mv.direction([0.0, 1.0])   # [] Point-, Point-

## 1. An ellipsoid held by its tangents

An ellipsoid's dual vanishes on exactly the planes that touch it, and sends each of them to its point of contact. Of the planes facing a given way, two touch the ellipsoid, one either side of its centre; the outward one is its tangent that way, whatever the scale or sign of the dual.

In [ ]:
def ellipsoid(semi_x: float, semi_y: float) -> Quadric:
    """The dual ellipsoid with the given semi-axes along x and y, centred at the origin: each axis
    direction paired with itself and weighted by its squared semi-axis, less the origin paired with
    itself."""
    return along_x * (Plane & along_x) * semi_x**2 + along_y * (Plane & along_y) * semi_y**2 - origin * (Plane & origin)


def motor(shift_x: float, shift_y: float, angle: float) -> Motor:
    """The rigid motion turning by an angle about the origin, then moving it by the given shifts."""
    translator = ((infinity ^ (mv.x * shift_x + mv.y * shift_y)) * -0.5).exp()
    rotor = (mv.xy * (-angle / 2.0)).exp()
    return translator * rotor


def tangent_plane(dual: Quadric, normal: Plane) -> Plane:
    """The tangent plane of a dual ellipsoid with the given outward normal, whatever its scale or sign.

    The planes with the normal's direction are `normal - infinity * offset`. Tangency is quadratic in the
    offset: `a * offset**2 - 2 * b * offset + c == 0`, with `a`, `b` and `c` the pairings of the plane at
    infinity and the normal through the dual. Its two roots lie either side of the centre's offset
    `b / a`; the outward one lies beyond it along the normal. Rescaling the dual rescales all three
    coefficients alike, so neither root moves.
    """
    unit = normal.normalized()                                                 # [...] Plane+
    a, b, c = infinity & dual(infinity), unit & dual(infinity), unit & dual(unit)   # [...] Scalar+, Scalar+, Scalar+
    return unit - infinity * (b / a + ((b * b - a * c) / (a * a)).square_root())


# An ellipsoid's semi-axes, the shift and turn that place it, and six ways its tangents face.
semi_axes, shift, placement_turn = (2.0, 1.2), (1.5, -0.8), np.radians(35)
facing = np.radians([0, 45, 90, 135, 210, 315])

placement = motor(*shift, placement_turn)
upright = ellipsoid(*semi_axes)                                                # [] Point- <- Plane+
dual = placement >> upright(placement << Plane)                                # [] Point- <- Plane+
# Its tangent planes facing each way: each contains its own point of contact.
normals = (mv.xy * (-facing / 2)).exp() >> mv.x                                # [ways] Plane+
tangents = tangent_plane(dual, normals)                                        # [ways] Plane+

In [ ]:
print("each tangent paired with its point of contact:", (tangents & dual(tangents)).to_array())

## 2. Three poses, one number

The second ellipsoid is turned and moved so that its tangent point with one normal lands on the first ellipsoid's tangent point with the opposite normal: touching. From there it is moved apart along the normal, and into the first. For each pose, the determinant of the blend is a cubic in the blend's share, fixed by four samples, and its peak between zero and one classifies the pose.

In [ ]:
# The first ellipsoid: semi-axes, centre and turn; the second: semi-axes and turn; the first's normal at the
# contact; and the second's offsets along it, apart, touching and overlapping.
first_semi, first_centre, first_turn = (2.0, 1.0), (-1.2, 0.0), np.radians(25)
second_semi, second_turn = (1.6, 0.9), np.radians(-35)
contact_normal = np.radians(22)
offsets = np.array([0.8, 0.0, -0.6])
# The offsets the second ellipsoid slides through, from apart into the first, and the duration of a frame.
slide_offsets = np.linspace(0.8, -0.6, 85)
frame_ms = 40
# The shares at which the blend's determinant, a cubic, is sampled to fix it, and the shares it is drawn at.
samples = [[0.0], [1.0], [2.0], [-1.0]]
n_shares = 500

In [ ]:
def blend(first: Quadric, second: Quadric, share: Scalar) -> Quadric:
    """The blend of two dual quadrics, the given share of the way from the first to the second."""
    return first * (1.0 - share) + second * share


def cubic_peak(values: Scalar) -> tuple[Scalar, Scalar]:
    """The maximum of a cubic in the share, sampled at 0, 1, 2 and -1 along the first axis.

    The cubic is concave between 0 and 1, so its maximum is the root of the derivative
    `3 * c3 * share**2 + 2 * c2 * share + c1` where the second derivative is negative. Written as
    `c1 / ((c2 * c2 - 3 * c3 * c1).square_root() - c2)`, that root stays finite as c3 vanishes.
    """
    y0, y1, y2, y3 = values
    c3 = (3.0 * y0 - 3.0 * y1 + y2 - y3) / 6.0
    c2 = -y0 + 0.5 * y1 + 0.5 * y3
    c1 = -0.5 * y0 + y1 - y2 / 6.0 - y3 / 3.0
    c0 = y0
    peak = c1 / ((c2 * c2 - 3.0 * c3 * c1).square_root() - c2)
    return peak, ((c3 * peak + c2) * peak + c1) * peak + c0


placement = motor(*first_centre, first_turn)
first_shape = ellipsoid(*first_semi)                                           # [] Point- <- Plane+
first_ellipsoid = placement >> first_shape(placement << Plane)                 # [] Point- <- Plane+
normal = (mv.xy * (-contact_normal / 2)).exp() >> mv.x                         # [] Plane+
second_ellipsoid = ellipsoid(*second_semi)                                     # [] Point- <- Plane+
turn = motor(0.0, 0.0, second_turn)
turned_second = turn >> second_ellipsoid(turn << Plane)                        # [] Point- <- Plane+
# The two contact points, at unit weight, and the motion that lays the one on the other.
target = first_ellipsoid(tangent_plane(first_ellipsoid, normal))               # [] Point-
start = turned_second(tangent_plane(turned_second, -normal))                   # [] Point-
displacement = target.normalized() - start.normalized()                        # [] Point-
translator = ((infinity ^ displacement.dual()) * -0.5).exp()                   # [] Motor+
touch = translator * turn                                                      # [] Motor+
# The second ellipsoid shifted along the normal from touching: apart, touching and overlapping.
shifts = ((infinity ^ (normal * offsets)) * -0.5).exp()                        # [poses] Motor+
poses = shifts * touch                                                         # [poses] Motor+
placed_second = poses >> second_ellipsoid(poses << Plane)                      # [poses] Point- <- Plane+
# Four samples of the blend's determinant fix the cubic.
parameter, maximum = cubic_peak(blend(first_ellipsoid, placed_second, mv.scalar(samples)[:, None]).dual().det())   # [poses] Scalar+, Scalar+

In [ ]:
print("share at the peak, per pose:", parameter.to_array())
print("peak determinant, apart, touching and overlapping:", maximum.to_array())

## 3. The contact, and the shapes between

At the touching pose the blend at its peak is singular: its last singular vector is a plane it sends to nothing, the tangent shared by both ellipsoids, and each ellipsoid sends it to the same point, the contact. In the animation the second ellipsoid slides along the normal from apart, through touching, into the first. While the pair is apart, the peak blend is a hyperboloid, green, whose branches part the two, and the midplane between their facing tangents separates them; the hyperboloid narrows onto the shared tangent as the pair closes, and at contact both tangents and their points of contact meet on it; once the pair overlaps, the peak blend is an ellipsoid, purple, bridging the overlap. Beside it, the determinant against the blend's share, with its peak, which falls through zero at contact.

In [ ]:
_, _, planes = blend(first_ellipsoid, placed_second[1], parameter[1]).dual().svd()
contact_plane = planes[-1].normalized()                                         # [] Plane+
contact_point = first_ellipsoid(contact_plane).normalized()                     # [] Point-
# For the animation: the first ellipsoid's tangent along the normal, and the shares the determinant is drawn at.
first_tangent = tangent_plane(first_ellipsoid, normal)                          # [] Plane+
shares = mv.scalar(np.linspace(0.001, 0.999, n_shares)[:, None])               # [shares] Scalar+


def slide(offsets: np.ndarray) -> Iterator[tuple[Quadric, Plane, Plane, Polarity, Scalar, Scalar, Scalar]]:
    """The second ellipsoid at each offset along the normal from touching, its tangent plane with the opposite
    normal and the midplane of the two tangents, the blend at the determinant's peak as a map from points to
    their polar planes, and the determinant along the blends with its peak."""
    for offset in offsets:
        shift = ((infinity ^ (normal * offset)) * -0.5).exp()                  # [] Motor+
        pose = shift * touch                                                   # [] Motor+
        placed_second = pose >> second_ellipsoid(pose << Plane)                 # [] Point- <- Plane+
        parameter, maximum = cubic_peak(blend(first_ellipsoid, placed_second, mv.scalar(samples)).dual().det())   # [] Scalar+, Scalar+
        second_tangent = tangent_plane(placed_second, -normal)                  # [] Plane+
        midplane = ((first_tangent - second_tangent) * 0.5).normalized()        # [] Plane+
        yield (placed_second, second_tangent, midplane, blend(first_ellipsoid, placed_second, parameter).inverse(),
               blend(first_ellipsoid, placed_second, shares).dual().det(), parameter, maximum)


show_animation(render.animate_collision(first_ellipsoid, first_tangent, shares, slide(slide_offsets)), "quadric_collision", frame_ms,
               forward_then_back=True)

In [ ]:
print("the shared tangent against each ellipsoid:", (contact_plane & first_ellipsoid(contact_plane)).to_array(), (contact_plane & placed_second[1](contact_plane)).to_array())
print("the second ellipsoid's contact point against the first's:", (placed_second[1](contact_plane).normalized() & contact_point).norm().to_array())

In [ ]:
# checks
# Every tangent contains its point of contact; the peak determinant is positive apart, zero touching and
# negative overlapping; the shared tangent touches both ellipsoids, at one point.
np.testing.assert_allclose((tangents & dual(tangents)).to_array(), 0.0, atol=1e-10)
assert maximum[0] > 0 and maximum[2] < 0
np.testing.assert_allclose(maximum[1].to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose((contact_plane & first_ellipsoid(contact_plane)).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose((placed_second[1](contact_plane).normalized() & contact_point).norm().to_array(), 0.0, atol=1e-12)